# 03 — Extract features from pretrained models and save feature files

This notebook loads the saved train/validation/test image split files and extracts features from each pretrained model separately.

For your 4-channel images, each channel is passed independently as fake RGB into the pretrained model. The four channel-wise feature vectors are concatenated for that model.

For example:

```text
resnet50 feature = concat(
    resnet50(channel_0),
    resnet50(channel_1),
    resnet50(channel_2),
    resnet50(channel_3)
)
```

Outputs are saved under:

```text
<PROJECT_ROOT>/Output/features/<model_name>/
    train_features.npy
    val_features.npy
    test_features.npy
```

In [1]:
from pathlib import Path
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

# TensorFlow is only needed for the CytoImageNet Keras model.
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Change this path if your shared folder is different.
PROJECT_ROOT = Path(r"..")
OUTPUT_DIR = PROJECT_ROOT / "output"

SPLIT_DIR = OUTPUT_DIR / "splits"
MODEL_DIR = OUTPUT_DIR / "downloaded_pretrained_models"
FEATURE_DIR = OUTPUT_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_WORKERS = 0
OVERWRITE_EXISTING = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Split folder:", SPLIT_DIR)
print("Model folder:", MODEL_DIR)
print("Feature folder:", FEATURE_DIR)


Device: cuda
Split folder: ..\output\splits
Model folder: ..\output\downloaded_pretrained_models
Feature folder: ..\output\features


In [2]:
X_splits = {
    "train": np.load(SPLIT_DIR / "X_train.npy", mmap_mode="r"),
    "val": np.load(SPLIT_DIR / "X_val.npy", mmap_mode="r"),
    "test": np.load(SPLIT_DIR / "X_test.npy", mmap_mode="r"),
}

class_names = np.load(SPLIT_DIR / "class_names.npy", allow_pickle=True)

for split_name, X_split in X_splits.items():
    print(split_name, X_split.shape, X_split.dtype)

print("Number of classes:", len(class_names))


train (82434, 4, 100, 100) float32
val (27478, 4, 100, 100) float32
test (27478, 4, 100, 100) float32
Number of classes: 32


In [3]:
class ImageArrayDataset(Dataset):
    def __init__(self, X_array):
        self.X_array = X_array

    def __len__(self):
        return len(self.X_array)

    def __getitem__(self, idx):
        x = np.asarray(self.X_array[idx], dtype=np.float32)
        x = np.clip(x, 0.0, 1.0)
        return torch.from_numpy(x)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [4]:
TORCH_MODEL_SPECS = {
    "resnet50": {
        "builder": models.resnet50,
        "weights": models.ResNet50_Weights.DEFAULT,
    },
    "efficientnet_b0": {
        "builder": models.efficientnet_b0,
        "weights": models.EfficientNet_B0_Weights.DEFAULT,
    },
    "densenet121": {
        "builder": models.densenet121,
        "weights": models.DenseNet121_Weights.DEFAULT,
    },
    "mobilenet_v3_large": {
        "builder": models.mobilenet_v3_large,
        "weights": models.MobileNet_V3_Large_Weights.DEFAULT,
    },
    "vit_b_16": {
        "builder": models.vit_b_16,
        "weights": models.ViT_B_16_Weights.DEFAULT,
    },
}


def replace_classifier_with_identity(model_name, model):
    if model_name == "resnet50":
        model.fc = nn.Identity()
    elif model_name == "efficientnet_b0":
        model.classifier = nn.Identity()
    elif model_name == "densenet121":
        model.classifier = nn.Identity()
    elif model_name == "mobilenet_v3_large":
        model.classifier = nn.Identity()
    elif model_name == "vit_b_16":
        model.heads = nn.Identity()
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return model


def load_torch_feature_model(model_name):
    spec = TORCH_MODEL_SPECS[model_name]
    weights = spec["weights"]
    state_path = MODEL_DIR / f"{model_name}_imagenet.pth"

    if state_path.exists():
        print(f"Loading local weights for {model_name}: {state_path}")
        model = spec["builder"](weights=None)
        state_dict = torch.load(state_path, map_location="cpu")
        model.load_state_dict(state_dict)
    else:
        print(f"Local weights not found. Downloading/loading {model_name} from torchvision.")
        model = spec["builder"](weights=weights)

    model = replace_classifier_with_identity(model_name, model)
    model.eval()
    model.to(device)

    preprocess = weights.transforms()

    return model, preprocess


In [5]:
def extract_torch_features_for_split(X_array, model, preprocess, batch_size=BATCH_SIZE):
    dataset = ImageArrayDataset(X_array)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    all_features = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            # batch shape: (B, 4, H, W), values in [0, 1]
            batch = batch.float()

            channel_features = []

            for channel_idx in range(batch.shape[1]):
                x_channel = batch[:, channel_idx:channel_idx + 1, :, :]
                x_rgb = x_channel.repeat(1, 3, 1, 1)

                # torchvision weight transforms handle resize/crop/normalization.
                x_rgb = preprocess(x_rgb)
                x_rgb = x_rgb.to(device, non_blocking=True)

                features = model(x_rgb)

                if isinstance(features, (tuple, list)):
                    features = features[0]

                features = torch.flatten(features, start_dim=1)
                channel_features.append(features.cpu().numpy())

            batch_features = np.concatenate(channel_features, axis=1)
            all_features.append(batch_features)

            if (batch_idx + 1) % 10 == 0:
                print(f"  processed {batch_idx + 1} batches")

    return np.concatenate(all_features, axis=0).astype(np.float32)


In [6]:
def extract_and_save_torch_model(model_name):
    model_out_dir = FEATURE_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    model, preprocess = load_torch_feature_model(model_name)

    for split_name, X_split in X_splits.items():
        out_file = model_out_dir / f"{split_name}_features.npy"

        if out_file.exists() and not OVERWRITE_EXISTING:
            print(f"[{model_name}] Skipping existing file:", out_file)
            continue

        print("=" * 80)
        print(f"[{model_name}] Extracting {split_name} features...")

        features = extract_torch_features_for_split(
            X_split,
            model=model,
            preprocess=preprocess,
            batch_size=BATCH_SIZE,
        )

        np.save(out_file, features)

        print(f"[{model_name}] Saved {split_name} features:", out_file)
        print(f"[{model_name}] Feature shape:", features.shape)

    del model
    clear_memory()


In [7]:
def load_cytoimagenet_feature_model():
    weights_path = MODEL_DIR / "cytoimagenet" / "efficientnetb0_weights-notop.h5"

    print("CytoImageNet weights path:", weights_path)

    if not weights_path.exists():
        raise FileNotFoundError(
            f"Cannot find CytoImageNet weights: {weights_path}. "
            "Run 01_model_download.ipynb first."
        )

    base_model = EfficientNetB0(
        include_top=False,
        weights=None,
        input_shape=(224, 224, 3),
    )

    base_model.load_weights(str(weights_path))

    feature_model = Model(
        inputs=base_model.input,
        outputs=GlobalAveragePooling2D()(base_model.output),
    )

    feature_model.trainable = False

    return feature_model


def prepare_cyto_batch(x_channel_batch):
    # x_channel_batch shape: (B, H, W), values normally in [0, 1]
    x = np.asarray(x_channel_batch, dtype=np.float32)
    x = np.clip(x, 0.0, 1.0)

    x = x[..., np.newaxis]          # (B, H, W, 1)
    x = np.repeat(x, 3, axis=-1)    # (B, H, W, 3)

    # Keras EfficientNet expects image-like values. In Keras EfficientNet,
    # preprocessing is part of the model, so use [0, 255]-scale input.
    x = x * 255.0

    x = tf.image.resize(x, (224, 224)).numpy()

    return x


def extract_cyto_features_for_split(X_array, feature_model, batch_size=BATCH_SIZE):
    all_channel_features = []

    for channel_idx in range(X_array.shape[1]):
        print(f"  CytoImageNet channel {channel_idx}")

        features_for_channel = []

        for start in range(0, len(X_array), batch_size):
            end = min(start + batch_size, len(X_array))

            x_channel_batch = np.asarray(
                X_array[start:end, channel_idx, :, :],
                dtype=np.float32,
            )

            x_batch = prepare_cyto_batch(x_channel_batch)

            batch_features = feature_model.predict(
                x_batch,
                batch_size=batch_size,
                verbose=0,
            )

            features_for_channel.append(batch_features.astype(np.float32))

        channel_features = np.concatenate(features_for_channel, axis=0)
        print("  channel feature shape:", channel_features.shape)
        all_channel_features.append(channel_features)

    return np.concatenate(all_channel_features, axis=1).astype(np.float32)


def extract_and_save_cytoimagenet():
    model_name = "cytoimagenet_efficientnetb0"
    model_out_dir = FEATURE_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    feature_model = load_cytoimagenet_feature_model()

    for split_name, X_split in X_splits.items():
        out_file = model_out_dir / f"{split_name}_features.npy"

        if out_file.exists() and not OVERWRITE_EXISTING:
            print(f"[{model_name}] Skipping existing file:", out_file)
            continue

        print("=" * 80)
        print(f"[{model_name}] Extracting {split_name} features...")

        features = extract_cyto_features_for_split(
            X_split,
            feature_model=feature_model,
            batch_size=BATCH_SIZE,
        )

        np.save(out_file, features)

        print(f"[{model_name}] Saved {split_name} features:", out_file)
        print(f"[{model_name}] Feature shape:", features.shape)

    del feature_model
    clear_memory()


In [8]:
# Choose which models to run.
# You can remove models from this list if feature extraction is too slow.

MODELS_TO_EXTRACT = [
    "resnet50",
    "efficientnet_b0",
    "densenet121",
    "mobilenet_v3_large",
    "vit_b_16",
    "cytoimagenet_efficientnetb0",
]

print("Models selected for feature extraction:")
for name in MODELS_TO_EXTRACT:
    print("-", name)


Models selected for feature extraction:
- resnet50
- efficientnet_b0
- densenet121
- mobilenet_v3_large
- vit_b_16
- cytoimagenet_efficientnetb0


In [9]:
for model_name in MODELS_TO_EXTRACT:
    print("\n" + "#" * 100)
    print("Feature extraction model:", model_name)
    print("#" * 100)

    if model_name == "cytoimagenet_efficientnetb0":
        extract_and_save_cytoimagenet()
    else:
        extract_and_save_torch_model(model_name)

print("All selected feature extraction jobs finished.")



####################################################################################################
Feature extraction model: resnet50
####################################################################################################
Loading local weights for resnet50: ..\output\downloaded_pretrained_models\resnet50_imagenet.pth
[resnet50] Extracting train features...
  processed 10 batches
  processed 20 batches
  processed 30 batches
  processed 40 batches
  processed 50 batches
  processed 60 batches
  processed 70 batches
  processed 80 batches
  processed 90 batches
  processed 100 batches
  processed 110 batches
  processed 120 batches
  processed 130 batches
  processed 140 batches
  processed 150 batches
  processed 160 batches
  processed 170 batches
  processed 180 batches
  processed 190 batches
  processed 200 batches
  processed 210 batches
  processed 220 batches
  processed 230 batches
  processed 240 batches
  processed 250 batches
  processed 260 batches
  processe

In [10]:
print("Saved feature folders:")

for model_folder in sorted(FEATURE_DIR.iterdir()):
    if not model_folder.is_dir():
        continue

    print("\n", model_folder.name)
    for path in sorted(model_folder.glob("*_features.npy")):
        arr = np.load(path, mmap_mode="r")
        print(" ", path.name, arr.shape)


Saved feature folders:

 cytoimagenet_efficientnetb0
  test_features.npy (27478, 5120)
  train_features.npy (82434, 5120)
  val_features.npy (27478, 5120)

 densenet121
  test_features.npy (27478, 4096)
  train_features.npy (82434, 4096)
  val_features.npy (27478, 4096)

 efficientnet_b0
  test_features.npy (27478, 5120)
  train_features.npy (82434, 5120)
  val_features.npy (27478, 5120)

 mobilenet_v3_large
  test_features.npy (27478, 3840)
  train_features.npy (82434, 3840)
  val_features.npy (27478, 3840)

 resnet50
  test_features.npy (27478, 8192)
  train_features.npy (82434, 8192)
  val_features.npy (27478, 8192)

 vit_b_16
  test_features.npy (27478, 3072)
  train_features.npy (82434, 3072)
  val_features.npy (27478, 3072)
